# 29. Prepare Streamlit Dashboard Assets

This notebook prepares lightweight dashboard-ready assets for the controlled 50-painting restoration evaluation.

The notebook does not run new experiments. It exports clean summary tables, case manifests, visual manifests, and JSON metadata that can be loaded directly by a Streamlit dashboard.

The goal is to make the dashboard fast, stable, and independent from the full experimental notebook chain.

## Dashboard asset strategy

The dashboard should not load every raw metric CSV from the project.

Instead, this notebook prepares curated assets under:

`outputs/dashboard/`

The dashboard assets cover:

- project overview,
- dataset and damage design,
- model stack,
- metric-region policy,
- final model comparison,
- Stable Diffusion uncertainty,
- visual case explorer,
- key findings,
- report links.

Large images are not embedded into JSON. The dashboard stores image paths and lets Streamlit load them only when needed.

## Expected dashboard sections

The prepared assets support the following dashboard structure:

1. Overview
2. Dataset and damage design
3. Model stack
4. Metric-region policy
5. Final model comparison
6. Stable Diffusion uncertainty
7. Visual case explorer
8. Key findings and thesis interpretation
9. Reports and reproducibility

This notebook focuses only on asset preparation. The Streamlit app itself can be created or updated separately.

In [1]:
from pathlib import Path
import sys
import json
from datetime import datetime

import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("Project root:", PROJECT_ROOT)
print("Source path:", src_path)

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Source path: D:\Masters\FH\Thesis\painting-restoration-eval\src


In [2]:
config_path = PROJECT_ROOT / "config" / "experiment_50_config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Config file not found: {config_path}")

with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

paths_cfg = config["paths"]

processed_metadata_dir = PROJECT_ROOT / paths_cfg["processed_metadata_dir"]
metrics_dir = PROJECT_ROOT / paths_cfg["metrics_dir"]
figures_dir = PROJECT_ROOT / paths_cfg["figures_dir"]
reports_dir = PROJECT_ROOT / paths_cfg.get("reports_dir", "outputs/reports")

dashboard_dir = PROJECT_ROOT / "outputs" / "dashboard"
dashboard_data_dir = dashboard_dir / "data"
dashboard_manifest_dir = dashboard_dir / "manifests"

dashboard_dir.mkdir(parents=True, exist_ok=True)
dashboard_data_dir.mkdir(parents=True, exist_ok=True)
dashboard_manifest_dir.mkdir(parents=True, exist_ok=True)

print("Processed metadata dir:", processed_metadata_dir)
print("Metrics dir:", metrics_dir)
print("Figures dir:", figures_dir)
print("Reports dir:", reports_dir)
print("Dashboard dir:", dashboard_dir)
print("Dashboard data dir:", dashboard_data_dir)
print("Dashboard manifest dir:", dashboard_manifest_dir)

Processed metadata dir: D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata
Metrics dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics
Figures dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\figures
Reports dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports
Dashboard dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard
Dashboard data dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data
Dashboard manifest dir: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests


In [3]:
dashboard_output_paths = {
    "overview_summary_json": dashboard_manifest_dir / "dashboard_overview_summary.json",
    "assets_manifest_json": dashboard_manifest_dir / "dashboard_assets_manifest.json",
    "key_findings_json": dashboard_manifest_dir / "dashboard_key_findings.json",
    "reports_manifest_json": dashboard_manifest_dir / "dashboard_reports_manifest.json",

    "dataset_summary_csv": dashboard_data_dir / "dashboard_dataset_summary.csv",
    "damage_summary_csv": dashboard_data_dir / "dashboard_damage_summary.csv",
    "model_stack_csv": dashboard_data_dir / "dashboard_model_stack.csv",
    "metric_policy_csv": dashboard_data_dir / "dashboard_metric_policy.csv",
    "model_win_summary_csv": dashboard_data_dir / "dashboard_model_win_summary.csv",
    "per_metric_winner_summary_csv": dashboard_data_dir / "dashboard_per_metric_winner_summary.csv",
    "model_comparison_cases_csv": dashboard_data_dir / "dashboard_model_comparison_cases.csv",
    "model_comparison_by_mask_type_csv": dashboard_data_dir / "dashboard_model_comparison_by_mask_type.csv",
    "model_comparison_by_category_csv": dashboard_data_dir / "dashboard_model_comparison_by_category.csv",

    "uncertainty_summary_csv": dashboard_data_dir / "dashboard_uncertainty_summary.csv",
    "uncertainty_cases_csv": dashboard_data_dir / "dashboard_uncertainty_cases.csv",
    "uncertainty_by_mask_type_csv": dashboard_data_dir / "dashboard_uncertainty_by_mask_type.csv",
    "uncertainty_by_category_csv": dashboard_data_dir / "dashboard_uncertainty_by_category.csv",
    "uncertainty_vs_performance_csv": dashboard_data_dir / "dashboard_uncertainty_vs_performance.csv",
    "uncertainty_quadrants_csv": dashboard_data_dir / "dashboard_uncertainty_quadrants.csv",

    "visual_cases_csv": dashboard_data_dir / "dashboard_visual_cases.csv",
}

for name, path in dashboard_output_paths.items():
    print(f"{name}: {path}")

overview_summary_json: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_overview_summary.json
assets_manifest_json: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_assets_manifest.json
key_findings_json: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_key_findings.json
reports_manifest_json: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_reports_manifest.json
dataset_summary_csv: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_dataset_summary.csv
damage_summary_csv: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_damage_summary.csv
model_stack_csv: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_model_stack.csv
metric_policy_csv: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_metric_policy.csv
model_win_summary_cs

In [4]:
source_paths = {
    # Final report synthesis outputs from Notebook 28
    "final_dataset_summary": metrics_dir / "final_controlled_50_dataset_summary.csv",
    "final_model_stack": metrics_dir / "final_controlled_50_model_stack_summary.csv",
    "final_metric_policy": metrics_dir / "final_controlled_50_metric_policy_summary.csv",
    "final_key_results": metrics_dir / "final_controlled_50_key_results_summary.csv",
    "final_model_win_summary": metrics_dir / "final_controlled_50_model_win_summary.csv",
    "final_per_metric_winner_summary": metrics_dir / "final_controlled_50_per_metric_winner_summary.csv",
    "final_uncertainty_summary": metrics_dir / "final_controlled_50_uncertainty_summary.csv",
    "final_sdxl_feasibility": metrics_dir / "final_controlled_50_sdxl_feasibility_summary.csv",
    "final_visual_cases": metrics_dir / "final_controlled_50_visual_cases.csv",
    "final_visual_case_summary": metrics_dir / "final_controlled_50_visual_case_summary.csv",

    # Refined comparison outputs
    "refined_comparison": metrics_dir / "comparison_unified_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_win_rates": metrics_dir / "comparison_win_rates_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_summary_by_mask_type": metrics_dir / "comparison_summary_by_mask_type_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_summary_by_category": metrics_dir / "comparison_summary_by_category_refined_opencv_lama_stable_diffusion_50.csv",
    "refined_metric_disagreement": metrics_dir / "comparison_metric_disagreement_cases_refined_opencv_lama_stable_diffusion_50.csv",

    # Uncertainty outputs
    "uncertainty_combined_by_case": metrics_dir / "stable_diffusion_uncertainty_combined_summary_by_case_50.csv",
    "uncertainty_combined_by_mask_type": metrics_dir / "stable_diffusion_uncertainty_combined_summary_by_mask_type_50.csv",
    "uncertainty_combined_by_category": metrics_dir / "stable_diffusion_uncertainty_combined_summary_by_category_50.csv",
    "uncertainty_vs_refined": metrics_dir / "stable_diffusion_uncertainty_vs_refined_performance_50.csv",
    "uncertainty_quadrants": metrics_dir / "stable_diffusion_uncertainty_performance_quadrants_50.csv",

    # Metadata
    "processed_metadata": processed_metadata_dir / "metadata_processed_clean.csv",
    "damage_metadata": processed_metadata_dir / "metadata_damaged_images.csv",

    # Reports
    "final_report": reports_dir / "final_controlled_50_evaluation_report.html",
    "refined_report": reports_dir / "opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html",
    "uncertainty_report": reports_dir / "stable_diffusion_uncertainty_report_50.html",
}

for name, path in source_paths.items():
    print(f"{name}: {path}")

final_dataset_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_dataset_summary.csv
final_model_stack: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_model_stack_summary.csv
final_metric_policy: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_metric_policy_summary.csv
final_key_results: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_key_results_summary.csv
final_model_win_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_model_win_summary.csv
final_per_metric_winner_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_per_metric_winner_summary.csv
final_uncertainty_summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\final_controlled_50_uncertainty_summary.csv
final_sdxl_feasibility: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\met

In [5]:
required_source_names = [
    "final_dataset_summary",
    "final_model_stack",
    "final_metric_policy",
    "final_key_results",
    "final_model_win_summary",
    "final_per_metric_winner_summary",
    "final_uncertainty_summary",
    "final_sdxl_feasibility",
    "final_visual_cases",
    "final_visual_case_summary",
    "refined_comparison",
    "refined_win_rates",
    "refined_summary_by_mask_type",
    "refined_summary_by_category",
    "refined_metric_disagreement",
    "uncertainty_combined_by_case",
    "uncertainty_combined_by_mask_type",
    "uncertainty_combined_by_category",
    "uncertainty_vs_refined",
    "uncertainty_quadrants",
    "processed_metadata",
    "damage_metadata",
    "final_report",
    "refined_report",
    "uncertainty_report",
]

missing_sources = [
    (name, source_paths[name])
    for name in required_source_names
    if not source_paths[name].exists()
]

if missing_sources:
    for name, path in missing_sources:
        print(f"Missing source {name}: {path}")
    raise FileNotFoundError("Some required dashboard source files are missing.")

print("All required dashboard source files exist.")

All required dashboard source files exist.


In [6]:
final_dataset_summary_df = pd.read_csv(source_paths["final_dataset_summary"])
final_model_stack_df = pd.read_csv(source_paths["final_model_stack"])
final_metric_policy_df = pd.read_csv(source_paths["final_metric_policy"])
final_key_results_df = pd.read_csv(source_paths["final_key_results"])
final_model_win_summary_df = pd.read_csv(source_paths["final_model_win_summary"])
final_per_metric_winner_summary_df = pd.read_csv(source_paths["final_per_metric_winner_summary"])
final_uncertainty_summary_df = pd.read_csv(source_paths["final_uncertainty_summary"])
final_sdxl_feasibility_df = pd.read_csv(source_paths["final_sdxl_feasibility"])
final_visual_cases_df = pd.read_csv(source_paths["final_visual_cases"])
final_visual_case_summary_df = pd.read_csv(source_paths["final_visual_case_summary"])

refined_comparison_df = pd.read_csv(source_paths["refined_comparison"])
refined_win_rates_df = pd.read_csv(source_paths["refined_win_rates"])
refined_summary_by_mask_type_df = pd.read_csv(source_paths["refined_summary_by_mask_type"])
refined_summary_by_category_df = pd.read_csv(source_paths["refined_summary_by_category"])
refined_metric_disagreement_df = pd.read_csv(source_paths["refined_metric_disagreement"])

uncertainty_combined_by_case_df = pd.read_csv(source_paths["uncertainty_combined_by_case"])
uncertainty_combined_by_mask_type_df = pd.read_csv(source_paths["uncertainty_combined_by_mask_type"])
uncertainty_combined_by_category_df = pd.read_csv(source_paths["uncertainty_combined_by_category"])
uncertainty_vs_refined_df = pd.read_csv(source_paths["uncertainty_vs_refined"])
uncertainty_quadrants_df = pd.read_csv(source_paths["uncertainty_quadrants"])

processed_metadata_df = pd.read_csv(source_paths["processed_metadata"])
damage_metadata_df = pd.read_csv(source_paths["damage_metadata"])

loaded_dashboard_sources = {
    "final_dataset_summary": final_dataset_summary_df,
    "final_model_stack": final_model_stack_df,
    "final_metric_policy": final_metric_policy_df,
    "final_key_results": final_key_results_df,
    "final_model_win_summary": final_model_win_summary_df,
    "final_visual_cases": final_visual_cases_df,
    "refined_comparison": refined_comparison_df,
    "uncertainty_combined_by_case": uncertainty_combined_by_case_df,
    "uncertainty_vs_refined": uncertainty_vs_refined_df,
    "processed_metadata": processed_metadata_df,
    "damage_metadata": damage_metadata_df,
}

for name, df in loaded_dashboard_sources.items():
    print(f"{name}: {df.shape}")

final_dataset_summary: (7, 4)
final_model_stack: (4, 8)
final_metric_policy: (6, 4)
final_key_results: (8, 4)
final_model_win_summary: (4, 6)
final_visual_cases: (110, 18)
refined_comparison: (200, 40)
uncertainty_combined_by_case: (40, 55)
uncertainty_vs_refined: (40, 75)
processed_metadata: (50, 43)
damage_metadata: (250, 19)


In [7]:
if len(processed_metadata_df) != 50:
    raise ValueError(f"Expected 50 processed paintings, found {len(processed_metadata_df)}.")

if len(damage_metadata_df) != 250:
    raise ValueError(f"Expected 250 damage rows, found {len(damage_metadata_df)}.")

if len(refined_comparison_df) != 200:
    raise ValueError(f"Expected 200 refined comparison rows, found {len(refined_comparison_df)}.")

if len(uncertainty_combined_by_case_df) != 40:
    raise ValueError(
        f"Expected 40 uncertainty cases, found {len(uncertainty_combined_by_case_df)}."
    )

if len(uncertainty_vs_refined_df) != 40:
    raise ValueError(
        f"Expected 40 uncertainty-performance rows, found {len(uncertainty_vs_refined_df)}."
    )

required_visual_columns = [
    "original_case_id",
    "category",
    "mask_type",
    "final_visual_source",
    "final_visual_reason",
    "final_figure_path",
    "final_figure_exists",
]

missing_visual_columns = [
    column for column in required_visual_columns
    if column not in final_visual_cases_df.columns
]

if missing_visual_columns:
    raise ValueError(f"Final visual cases missing columns: {missing_visual_columns}")

if final_visual_cases_df["final_figure_exists"].sum() == 0:
    raise ValueError("Final visual cases have no available figures.")

print("Dashboard source validation gates passed.")

Dashboard source validation gates passed.


## Dashboard asset generation

This section creates clean dashboard-ready CSV and JSON files.

The dashboard assets are designed to be:

- lightweight,
- easy to load,
- stable across reruns,
- independent from raw notebook outputs,
- organized under `outputs/dashboard/`.

The dashboard should use these exported assets rather than reading directly from every experimental output file.

In [8]:
def to_project_relative_path(path_value: str | Path) -> str:
    path = Path(str(path_value))

    if not path.is_absolute():
        return path.as_posix()

    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def resolve_project_path(path_value: str | Path) -> Path:
    path = Path(str(path_value))

    if path.is_absolute():
        return path

    return PROJECT_ROOT / path


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)


def dataframe_records_for_json(df: pd.DataFrame) -> list[dict]:
    clean_df = df.copy()
    clean_df = clean_df.replace({np.nan: None})
    return clean_df.to_dict("records")


print("Dashboard path and JSON helpers ready.")

Dashboard path and JSON helpers ready.


In [9]:
dashboard_generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

total_paintings = int(len(processed_metadata_df))
total_damage_cases = int(len(damage_metadata_df))
total_non_zero_cases = int(len(refined_comparison_df))
total_uncertainty_cases = int(len(uncertainty_combined_by_case_df))

model_win_lookup = {
    row["model_name"]: row
    for _, row in final_model_win_summary_df.iterrows()
}

lama_majority_cases = int(model_win_lookup.get("lama", {}).get("majority_vote_cases", 0))
opencv_majority_cases = int(model_win_lookup.get("opencv_telea", {}).get("majority_vote_cases", 0))
sd_majority_cases = int(
    model_win_lookup.get("stable_diffusion_inpainting", {}).get("majority_vote_cases", 0)
)

highest_uncertainty_row = uncertainty_combined_by_case_df.sort_values(
    "combined_uncertainty_index",
    ascending=False,
).iloc[0]

overview_summary = {
    "generated_at": dashboard_generated_at,
    "project_title": "Trustworthy Evaluation Frameworks for AI-Assisted Painting Restoration",
    "controlled_subset": {
        "paintings": total_paintings,
        "painting_categories": int(processed_metadata_df["category"].nunique()),
        "damage_cases": total_damage_cases,
        "non_zero_comparison_cases": total_non_zero_cases,
    },
    "models": {
        "fully_evaluated": [
            "opencv_telea",
            "lama",
            "stable_diffusion_inpainting",
        ],
        "feasibility_audited": [
            "sdxl_inpainting",
        ],
    },
    "refined_comparison": {
        "lama_majority_cases": lama_majority_cases,
        "opencv_telea_majority_cases": opencv_majority_cases,
        "stable_diffusion_inpainting_majority_cases": sd_majority_cases,
        "total_non_zero_cases": total_non_zero_cases,
    },
    "uncertainty_analysis": {
        "cases": total_uncertainty_cases,
        "seed_outputs": int(40 * 4),
        "seeds_per_case": 4,
        "highest_uncertainty_case": str(highest_uncertainty_row["original_case_id"]),
        "highest_uncertainty_case_mask_type": str(highest_uncertainty_row["mask_type"]),
        "highest_uncertainty_index": float(highest_uncertainty_row["combined_uncertainty_index"]),
    },
    "central_claim": "Visual plausibility is not the same as restoration trustworthiness.",
    "main_report": to_project_relative_path(source_paths["final_report"]),
}

write_json(dashboard_output_paths["overview_summary_json"], overview_summary)

print("Saved dashboard overview summary:", dashboard_output_paths["overview_summary_json"])
overview_summary

Saved dashboard overview summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_overview_summary.json


{'generated_at': '2026-07-06 18:07:37',
 'project_title': 'Trustworthy Evaluation Frameworks for AI-Assisted Painting Restoration',
 'controlled_subset': {'paintings': 50,
  'painting_categories': 5,
  'damage_cases': 250,
  'non_zero_comparison_cases': 200},
 'models': {'fully_evaluated': ['opencv_telea',
   'lama',
   'stable_diffusion_inpainting'],
  'feasibility_audited': ['sdxl_inpainting']},
 'refined_comparison': {'lama_majority_cases': 155,
  'opencv_telea_majority_cases': 21,
  'stable_diffusion_inpainting_majority_cases': 1,
  'total_non_zero_cases': 200},
 'uncertainty_analysis': {'cases': 40,
  'seed_outputs': 160,
  'seeds_per_case': 4,
  'highest_uncertainty_case': 'p011_loss_large',
  'highest_uncertainty_case_mask_type': 'loss_large',
  'highest_uncertainty_index': 0.9834162335148644},
 'central_claim': 'Visual plausibility is not the same as restoration trustworthiness.',
 'main_report': 'outputs/reports/final_controlled_50_evaluation_report.html'}

In [10]:
dashboard_dataset_summary_df = final_dataset_summary_df.copy()

dashboard_damage_summary_df = (
    damage_metadata_df
    .groupby("mask_type", dropna=False)
    .agg(
        cases=("case_id", "count"),
        paintings=("painting_id", "nunique"),
    )
    .reset_index()
    .sort_values("mask_type")
)

if "category" in damage_metadata_df.columns:
    dashboard_damage_by_category_df = (
        damage_metadata_df
        .groupby(["category", "mask_type"], dropna=False)
        .agg(
            cases=("case_id", "count"),
            paintings=("painting_id", "nunique"),
        )
        .reset_index()
        .sort_values(["category", "mask_type"])
    )
else:
    dashboard_damage_by_category_df = pd.DataFrame()

dashboard_damage_by_category_path = dashboard_data_dir / "dashboard_damage_by_category.csv"

dashboard_dataset_summary_df.to_csv(
    dashboard_output_paths["dataset_summary_csv"],
    index=False,
)

dashboard_damage_summary_df.to_csv(
    dashboard_output_paths["damage_summary_csv"],
    index=False,
)

dashboard_damage_by_category_df.to_csv(
    dashboard_damage_by_category_path,
    index=False,
)

print("Saved dataset summary:", dashboard_output_paths["dataset_summary_csv"])
print("Saved damage summary:", dashboard_output_paths["damage_summary_csv"])
print("Saved damage by category:", dashboard_damage_by_category_path)

display(dashboard_dataset_summary_df)
display(dashboard_damage_summary_df)

Saved dataset summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_dataset_summary.csv
Saved damage summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_damage_summary.csv
Saved damage by category: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_damage_by_category.csv


,section,item,value,description
0,dataset,total_paintings,50,Controlled painting subset used for the final ...
1,dataset,painting_categories,5,Five controlled painting categories.
2,dataset,paintings_per_category,10,Each category contributes ten paintings.
3,damage,total_damage_cases,250,50 paintings × 5 mask types.
4,damage,non_zero_damage_cases,200,Cases used for local restoration comparison.
5,damage,zero_control_cases,50,No-damage control cases.
6,damage,mask_types,5,"zero_control, scratch_thin, loss_small, loss_l..."


,mask_type,cases,paintings
0,loss_large,50,50
1,loss_small,50,50
2,mixed_damage,50,50
3,scratch_thin,50,50
4,zero_control,50,50


In [11]:
dashboard_model_stack_df = final_model_stack_df.copy()
dashboard_metric_policy_df = final_metric_policy_df.copy()

dashboard_model_stack_df.to_csv(
    dashboard_output_paths["model_stack_csv"],
    index=False,
)

dashboard_metric_policy_df.to_csv(
    dashboard_output_paths["metric_policy_csv"],
    index=False,
)

print("Saved model stack:", dashboard_output_paths["model_stack_csv"])
print("Saved metric policy:", dashboard_output_paths["metric_policy_csv"])

display(dashboard_model_stack_df)
display(dashboard_metric_policy_df)

Saved model stack: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_model_stack.csv
Saved metric policy: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_metric_policy.csv


,model_name,display_name,model_type,evaluation_status,cases_restored,non_zero_cases_compared,role_in_framework,main_interpretation
0,opencv_telea,OpenCV Telea,classical inpainting baseline,fully_evaluated,250,200,deterministic classical baseline,Useful baseline for local interpolation behavi...
1,lama,LaMa,deep learning inpainting model,fully_evaluated,250,200,strong reference-based inpainting model,Dominated the refined reference-based metric c...
2,stable_diffusion_inpainting,Stable Diffusion Inpainting,latent diffusion inpainting model,fully_evaluated_plus_uncertainty_analysis,250,200,generative restoration candidate,Can produce visually plausible completions but...
3,sdxl_inpainting,SDXL Inpainting,larger latent diffusion inpainting model,feasibility_audit_only,smoke/probe only,0,excluded local full-evaluation candidate,Excluded from full local evaluation due runtim...


,metric,final_local_region,reason,used_in_final_comparison
0,MSE improvement,masked_region,Direct pixel-error metric; suitable for sparse...,True
1,PSNR improvement,masked_region,Derived from pixel error; suitable for sparse ...,True
2,SSIM improvement,mask_bbox_crop,Structural metric requiring image-like local c...,True
3,LPIPS improvement,mask_bbox_crop,Perceptual metric requiring image-like input.,True
4,CLIP similarity improvement,mask_bbox_crop,Feature metric requiring image-like crop aroun...,True
5,DINOv2 similarity improvement,mask_bbox_crop,Feature metric requiring image-like crop aroun...,True


In [12]:
dashboard_model_win_summary_df = final_model_win_summary_df.copy()
dashboard_per_metric_winner_summary_df = final_per_metric_winner_summary_df.copy()

dashboard_model_comparison_cases_columns = [
    "case_id",
    "painting_id",
    "mask_id",
    "mask_type",
    "category",
    "title",
    "overall_metric_vote",
    "metric_ties",
    "mixed_metric_outcome",
    "opencv_telea_metric_wins",
    "lama_metric_wins",
    "stable_diffusion_inpainting_metric_wins",
]

available_model_comparison_columns = [
    column for column in dashboard_model_comparison_cases_columns
    if column in refined_comparison_df.columns
]

dashboard_model_comparison_cases_df = refined_comparison_df[
    available_model_comparison_columns
].copy()

# Add compact model label columns for easier Streamlit filtering.
winner_display_map = {
    "opencv_telea": "OpenCV Telea",
    "lama": "LaMa",
    "stable_diffusion_inpainting": "Stable Diffusion Inpainting",
}

dashboard_model_comparison_cases_df["overall_metric_vote_display"] = (
    dashboard_model_comparison_cases_df["overall_metric_vote"]
    .map(winner_display_map)
    .fillna(dashboard_model_comparison_cases_df["overall_metric_vote"])
)

dashboard_model_win_summary_df.to_csv(
    dashboard_output_paths["model_win_summary_csv"],
    index=False,
)

dashboard_per_metric_winner_summary_df.to_csv(
    dashboard_output_paths["per_metric_winner_summary_csv"],
    index=False,
)

dashboard_model_comparison_cases_df.to_csv(
    dashboard_output_paths["model_comparison_cases_csv"],
    index=False,
)

refined_summary_by_mask_type_df.to_csv(
    dashboard_output_paths["model_comparison_by_mask_type_csv"],
    index=False,
)

refined_summary_by_category_df.to_csv(
    dashboard_output_paths["model_comparison_by_category_csv"],
    index=False,
)

print("Saved model win summary:", dashboard_output_paths["model_win_summary_csv"])
print("Saved per-metric winner summary:", dashboard_output_paths["per_metric_winner_summary_csv"])
print("Saved model comparison cases:", dashboard_output_paths["model_comparison_cases_csv"])
print("Saved model comparison by mask type:", dashboard_output_paths["model_comparison_by_mask_type_csv"])
print("Saved model comparison by category:", dashboard_output_paths["model_comparison_by_category_csv"])

display(dashboard_model_comparison_cases_df.head())

Saved model win summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_model_win_summary.csv
Saved per-metric winner summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_per_metric_winner_summary.csv
Saved model comparison cases: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_model_comparison_cases.csv
Saved model comparison by mask type: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_model_comparison_by_mask_type.csv
Saved model comparison by category: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_model_comparison_by_category.csv


,case_id,painting_id,mask_id,mask_type,category,title,overall_metric_vote,metric_ties,mixed_metric_outcome,opencv_telea_metric_wins,lama_metric_wins,stable_diffusion_inpainting_metric_wins,overall_metric_vote_display
0,p001_loss_large,p001,p001_loss_large,loss_large,portrait_figure,Juan de Pareja,lama,0,False,0,6,0,LaMa
1,p001_loss_small,p001,p001_loss_small,loss_small,portrait_figure,Juan de Pareja,lama,0,True,1,5,0,LaMa
2,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,portrait_figure,Juan de Pareja,tie_lama_opencv_telea,0,True,3,3,0,tie_lama_opencv_telea
3,p001_scratch_thin,p001,p001_scratch_thin,scratch_thin,portrait_figure,Juan de Pareja,lama,0,True,1,5,0,LaMa
4,p002_loss_large,p002,p002_loss_large,loss_large,portrait_figure,Madame X (Madame Pierre Gautreau),lama,0,False,0,6,0,LaMa


In [13]:
dashboard_uncertainty_summary_df = final_uncertainty_summary_df.copy()

uncertainty_case_columns = [
    "original_case_id",
    "painting_id",
    "mask_id",
    "mask_type",
    "category",
    "title",
    "uncertainty_selection_role",
    "combined_uncertainty_index",
    "uncertainty_rank_high_to_low",
    "masked_std_mean",
    "masked_mad_mean",
    "bbox_std_mean",
    "mean_pairwise_lpips",
    "mean_clip_uncertainty_distance",
    "mean_dinov2_uncertainty_distance",
]

available_uncertainty_case_columns = [
    column for column in uncertainty_case_columns
    if column in uncertainty_combined_by_case_df.columns
]

dashboard_uncertainty_cases_df = uncertainty_combined_by_case_df[
    available_uncertainty_case_columns
].copy()

dashboard_uncertainty_vs_performance_columns = [
    "original_case_id",
    "category",
    "mask_type",
    "combined_uncertainty_index",
    "refined_stable_diffusion_metric_wins",
    "refined_overall_metric_vote",
    "uncertainty_performance_quadrant",
]

available_uncertainty_vs_performance_columns = [
    column for column in dashboard_uncertainty_vs_performance_columns
    if column in uncertainty_vs_refined_df.columns
]

dashboard_uncertainty_vs_performance_df = uncertainty_vs_refined_df[
    available_uncertainty_vs_performance_columns
].copy()

dashboard_uncertainty_summary_df.to_csv(
    dashboard_output_paths["uncertainty_summary_csv"],
    index=False,
)

dashboard_uncertainty_cases_df.to_csv(
    dashboard_output_paths["uncertainty_cases_csv"],
    index=False,
)

uncertainty_combined_by_mask_type_df.to_csv(
    dashboard_output_paths["uncertainty_by_mask_type_csv"],
    index=False,
)

uncertainty_combined_by_category_df.to_csv(
    dashboard_output_paths["uncertainty_by_category_csv"],
    index=False,
)

dashboard_uncertainty_vs_performance_df.to_csv(
    dashboard_output_paths["uncertainty_vs_performance_csv"],
    index=False,
)

uncertainty_quadrants_df.to_csv(
    dashboard_output_paths["uncertainty_quadrants_csv"],
    index=False,
)

print("Saved uncertainty summary:", dashboard_output_paths["uncertainty_summary_csv"])
print("Saved uncertainty cases:", dashboard_output_paths["uncertainty_cases_csv"])
print("Saved uncertainty by mask type:", dashboard_output_paths["uncertainty_by_mask_type_csv"])
print("Saved uncertainty by category:", dashboard_output_paths["uncertainty_by_category_csv"])
print("Saved uncertainty vs performance:", dashboard_output_paths["uncertainty_vs_performance_csv"])
print("Saved uncertainty quadrants:", dashboard_output_paths["uncertainty_quadrants_csv"])

display(dashboard_uncertainty_cases_df.head())

Saved uncertainty summary: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_uncertainty_summary.csv
Saved uncertainty cases: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_uncertainty_cases.csv
Saved uncertainty by mask type: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_uncertainty_by_mask_type.csv
Saved uncertainty by category: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_uncertainty_by_category.csv
Saved uncertainty vs performance: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_uncertainty_vs_performance.csv
Saved uncertainty quadrants: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_uncertainty_quadrants.csv


,original_case_id,painting_id,mask_id,mask_type,category,title,uncertainty_selection_role,combined_uncertainty_index,uncertainty_rank_high_to_low,masked_std_mean,masked_mad_mean,bbox_std_mean,mean_pairwise_lpips,mean_clip_uncertainty_distance,mean_dinov2_uncertainty_distance
0,p011_loss_large,p011,p011_loss_large,loss_large,landscape_natural,View of Haarlem and the Haarlemmer Meer,relatively_weak_or_disagreement_prone_stable_d...,0.983416,1,0.149984,0.133327,0.081859,0.447396,0.232996,0.602260
1,p009_loss_large,p009,p009_loss_large,loss_large,portrait_figure,Madame Roulin and Her Baby,relatively_strong_stable_diffusion,0.719251,2,0.147536,0.129001,0.089974,0.372464,0.076807,0.154241
2,p031_loss_large,p031,p031_loss_large,loss_large,abstraction_surrealism,Improvisation No. 30 (Cannons),relatively_weak_or_disagreement_prone_stable_d...,0.562587,3,0.108735,0.095317,0.067748,0.318111,0.066506,0.255413
3,p018_loss_large,p018,p018_loss_large,loss_large,landscape_natural,Classical Landscape with Figures,relatively_strong_stable_diffusion,0.528796,4,0.088408,0.077121,0.048079,0.288359,0.120410,0.388862
4,p026_loss_large,p026,p026_loss_large,loss_large,architecture_structured,View of a Village along a River,relatively_weak_or_disagreement_prone_stable_d...,0.484108,5,0.083458,0.072228,0.053656,0.345339,0.094074,0.233291


In [14]:
dashboard_visual_cases_df = final_visual_cases_df.copy()

# Keep only useful and safe dashboard columns.
visual_case_columns = [
    "original_case_id",
    "case_id",
    "painting_id",
    "category",
    "mask_type",
    "title",
    "final_visual_source",
    "final_visual_reason",
    "final_figure_path",
    "final_figure_exists",
    "combined_uncertainty_index",
    "masked_std_mean",
    "mean_pairwise_lpips",
    "mean_dinov2_uncertainty_distance",
    "refined_stable_diffusion_metric_wins",
    "uncertainty_performance_quadrant",
    "overall_metric_vote",
    "refined_overall_metric_vote",
]

available_visual_case_columns = [
    column for column in visual_case_columns
    if column in dashboard_visual_cases_df.columns
]

dashboard_visual_cases_df = dashboard_visual_cases_df[
    available_visual_case_columns
].copy()

# Normalize figure paths to project-relative paths.
if "final_figure_path" in dashboard_visual_cases_df.columns:
    dashboard_visual_cases_df["final_figure_path"] = dashboard_visual_cases_df[
        "final_figure_path"
    ].fillna("").map(
        lambda path_value: to_project_relative_path(path_value)
        if str(path_value).strip()
        else ""
    )

dashboard_visual_cases_df["dashboard_figure_available"] = dashboard_visual_cases_df[
    "final_figure_path"
].map(
    lambda path_value: bool(str(path_value).strip()) and resolve_project_path(path_value).exists()
)

dashboard_visual_cases_df.to_csv(
    dashboard_output_paths["visual_cases_csv"],
    index=False,
)

print("Saved visual cases:", dashboard_output_paths["visual_cases_csv"])
print("Visual case rows:", len(dashboard_visual_cases_df))
print("Available figures:", int(dashboard_visual_cases_df["dashboard_figure_available"].sum()))

display(dashboard_visual_cases_df.head(20))

Saved visual cases: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\data\dashboard_visual_cases.csv
Visual case rows: 110
Available figures: 65


,original_case_id,case_id,painting_id,category,mask_type,title,final_visual_source,final_visual_reason,final_figure_path,final_figure_exists,combined_uncertainty_index,masked_std_mean,mean_pairwise_lpips,mean_dinov2_uncertainty_distance,refined_stable_diffusion_metric_wins,uncertainty_performance_quadrant,overall_metric_vote,refined_overall_metric_vote,dashboard_figure_available
0,p039_scratch_thin,p039_scratch_thin,p039,abstraction_surrealism,scratch_thin,Painting with Troika,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False
1,p026_loss_small,p026_loss_small,p026,architecture_structured,loss_small,View of a Village along a River,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False
2,p027_loss_small,p027_loss_small,p027,architecture_structured,loss_small,Italian Landscape with the Ponte Lucano over t...,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False
3,p030_loss_small,p030_loss_small,p030,architecture_structured,loss_small,Interior with Five Women,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False
4,p041_loss_large,p041_loss_large,p041,high_texture_brushwork,loss_large,Wheat Field with Cypresses,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False
5,p041_loss_small,p041_loss_small,p041,high_texture_brushwork,loss_small,Wheat Field with Cypresses,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False
6,p049_loss_small,p049_loss_small,p049,high_texture_brushwork,loss_small,Peonies,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False
7,p050_loss_small,p050_loss_small,p050,high_texture_brushwork,loss_small,Landscape with a Sunlit Stream,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False
8,p041_mixed_damage,p041_mixed_damage,p041,high_texture_brushwork,mixed_damage,Wheat Field with Cypresses,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False
9,p050_mixed_damage,p050_mixed_damage,p050,high_texture_brushwork,mixed_damage,Landscape with a Sunlit Stream,refined_model_comparison,old_vs_refined_vote_changed,,False,NaN,NaN,NaN,NaN,NaN,NaN,tie_lama_opencv_telea,NaN,False


In [15]:
key_findings_payload = {
    "generated_at": dashboard_generated_at,
    "central_claim": "Visual plausibility is not the same as restoration trustworthiness.",
    "findings": dataframe_records_for_json(final_key_results_df),
}

write_json(
    dashboard_output_paths["key_findings_json"],
    key_findings_payload,
)

print("Saved key findings JSON:", dashboard_output_paths["key_findings_json"])
key_findings_payload

Saved key findings JSON: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_key_findings.json


{'generated_at': '2026-07-06 18:07:37',
 'central_claim': 'Visual plausibility is not the same as restoration trustworthiness.',
 'findings': [{'finding_id': 'F1',
   'finding': 'Balanced controlled benchmark completed',
   'evidence': '50 paintings, 5 painting categories, 250 damage cases, 200 non-zero local comparison cases.',
   'thesis_interpretation': 'The experiment provides a controlled basis for comparing restoration behavior across categories and damage types.'},
  {'finding_id': 'F2',
   'finding': 'LaMa dominates refined reference-based comparison',
   'evidence': 'LaMa won the refined majority vote in 155/200 non-zero cases.',
   'thesis_interpretation': 'Deep inpainting can outperform both classical interpolation and diffusion generation under reference-based metrics.'},
  {'finding_id': 'F3',
   'finding': 'OpenCV Telea remains useful as deterministic baseline',
   'evidence': 'OpenCV Telea won the refined majority vote in 21/200 non-zero cases.',
   'thesis_interpretatio

In [16]:
reports_manifest = {
    "generated_at": dashboard_generated_at,
    "reports": [
        {
            "report_id": "final_controlled_50",
            "title": "Final Controlled 50-Painting Evaluation Report",
            "path": to_project_relative_path(source_paths["final_report"]),
            "description": "Main consolidated experimental report.",
            "recommended_primary": True,
        },
        {
            "report_id": "refined_model_comparison",
            "title": "Refined OpenCV-LaMa-Stable Diffusion Comparison Report",
            "path": to_project_relative_path(source_paths["refined_report"]),
            "description": "Detailed refined metric-region comparison report.",
            "recommended_primary": False,
        },
        {
            "report_id": "stable_diffusion_uncertainty",
            "title": "Stable Diffusion Uncertainty Report",
            "path": to_project_relative_path(source_paths["uncertainty_report"]),
            "description": "Detailed multi-seed Stable Diffusion uncertainty report.",
            "recommended_primary": False,
        },
    ],
}

write_json(
    dashboard_output_paths["reports_manifest_json"],
    reports_manifest,
)

print("Saved reports manifest:", dashboard_output_paths["reports_manifest_json"])
reports_manifest

Saved reports manifest: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_reports_manifest.json


{'generated_at': '2026-07-06 18:07:37',
 'reports': [{'report_id': 'final_controlled_50',
   'title': 'Final Controlled 50-Painting Evaluation Report',
   'path': 'outputs/reports/final_controlled_50_evaluation_report.html',
   'description': 'Main consolidated experimental report.',
   'recommended_primary': True},
  {'report_id': 'refined_model_comparison',
   'title': 'Refined OpenCV-LaMa-Stable Diffusion Comparison Report',
   'path': 'outputs/reports/opencv_lama_stable_diffusion_refined_metric_comparison_report_50.html',
   'description': 'Detailed refined metric-region comparison report.',
   'recommended_primary': False},
  {'report_id': 'stable_diffusion_uncertainty',
   'title': 'Stable Diffusion Uncertainty Report',
   'path': 'outputs/reports/stable_diffusion_uncertainty_report_50.html',
   'description': 'Detailed multi-seed Stable Diffusion uncertainty report.',
   'recommended_primary': False}]}

In [17]:
dashboard_asset_entries = []

for asset_name, path in dashboard_output_paths.items():
    if path.exists():
        dashboard_asset_entries.append(
            {
                "asset_name": asset_name,
                "path": to_project_relative_path(path),
                "file_type": path.suffix.lower().replace(".", ""),
                "size_bytes": path.stat().st_size,
            }
        )

dashboard_assets_manifest = {
    "generated_at": dashboard_generated_at,
    "dashboard_dir": to_project_relative_path(dashboard_dir),
    "data_dir": to_project_relative_path(dashboard_data_dir),
    "manifest_dir": to_project_relative_path(dashboard_manifest_dir),
    "assets": dashboard_asset_entries,
    "recommended_dashboard_sections": [
        "Overview",
        "Dataset and damage design",
        "Model stack",
        "Metric-region policy",
        "Final model comparison",
        "Stable Diffusion uncertainty",
        "Visual case explorer",
        "Key findings",
        "Reports and reproducibility",
    ],
}

write_json(
    dashboard_output_paths["assets_manifest_json"],
    dashboard_assets_manifest,
)

print("Saved dashboard assets manifest:", dashboard_output_paths["assets_manifest_json"])
print("Dashboard asset count:", len(dashboard_asset_entries))

display(pd.DataFrame(dashboard_asset_entries))

Saved dashboard assets manifest: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_assets_manifest.json
Dashboard asset count: 19


,asset_name,path,file_type,size_bytes
0,overview_summary_json,outputs/dashboard/manifests/dashboard_overview...,json,1111
1,key_findings_json,outputs/dashboard/manifests/dashboard_key_find...,json,3042
2,reports_manifest_json,outputs/dashboard/manifests/dashboard_reports_...,json,1036
3,dataset_summary_csv,outputs/dashboard/data/dashboard_dataset_summa...,csv,552
4,damage_summary_csv,outputs/dashboard/data/dashboard_damage_summar...,csv,123
5,model_stack_csv,outputs/dashboard/data/dashboard_model_stack.csv,csv,1040
6,metric_policy_csv,outputs/dashboard/data/dashboard_metric_policy...,csv,711
7,model_win_summary_csv,outputs/dashboard/data/dashboard_model_win_sum...,csv,320
8,per_metric_winner_summary_csv,outputs/dashboard/data/dashboard_per_metric_wi...,csv,736
9,model_comparison_cases_csv,outputs/dashboard/data/dashboard_model_compari...,csv,26529


In [18]:
expected_dashboard_files = list(dashboard_output_paths.values())

missing_dashboard_files = [
    str(path)
    for path in expected_dashboard_files
    if not path.exists()
]

if missing_dashboard_files:
    raise FileNotFoundError(
        "Missing dashboard output files:\n"
        + "\n".join(missing_dashboard_files)
    )

# Reload critical dashboard assets.
saved_dashboard_overview = json.loads(
    dashboard_output_paths["overview_summary_json"].read_text(encoding="utf-8")
)

saved_dashboard_assets_manifest = json.loads(
    dashboard_output_paths["assets_manifest_json"].read_text(encoding="utf-8")
)

saved_dashboard_model_cases_df = pd.read_csv(
    dashboard_output_paths["model_comparison_cases_csv"]
)

saved_dashboard_uncertainty_cases_df = pd.read_csv(
    dashboard_output_paths["uncertainty_cases_csv"]
)

saved_dashboard_visual_cases_df = pd.read_csv(
    dashboard_output_paths["visual_cases_csv"]
)

if saved_dashboard_overview["controlled_subset"]["paintings"] != 50:
    raise ValueError("Dashboard overview painting count is not 50.")

if len(saved_dashboard_model_cases_df) != 200:
    raise ValueError(
        f"Expected 200 dashboard model comparison cases, found {len(saved_dashboard_model_cases_df)}."
    )

if len(saved_dashboard_uncertainty_cases_df) != 40:
    raise ValueError(
        f"Expected 40 dashboard uncertainty cases, found {len(saved_dashboard_uncertainty_cases_df)}."
    )

if len(saved_dashboard_visual_cases_df) == 0:
    raise ValueError("Dashboard visual cases table is empty.")

if saved_dashboard_visual_cases_df["dashboard_figure_available"].sum() == 0:
    raise ValueError("Dashboard visual cases contain no available figures.")

if len(saved_dashboard_assets_manifest["assets"]) < 10:
    raise ValueError("Dashboard assets manifest contains too few assets.")

print("Dashboard asset validation gates passed.")
print("Dashboard overview:", dashboard_output_paths["overview_summary_json"])
print("Dashboard assets manifest:", dashboard_output_paths["assets_manifest_json"])
print("Dashboard model comparison rows:", len(saved_dashboard_model_cases_df))
print("Dashboard uncertainty rows:", len(saved_dashboard_uncertainty_cases_df))
print("Dashboard visual case rows:", len(saved_dashboard_visual_cases_df))
print("Dashboard available figures:", int(saved_dashboard_visual_cases_df["dashboard_figure_available"].sum()))
print("Dashboard asset count:", len(saved_dashboard_assets_manifest["assets"]))

Dashboard asset validation gates passed.
Dashboard overview: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_overview_summary.json
Dashboard assets manifest: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\dashboard\manifests\dashboard_assets_manifest.json
Dashboard model comparison rows: 200
Dashboard uncertainty rows: 40
Dashboard visual case rows: 110
Dashboard available figures: 65
Dashboard asset count: 19


## Notebook 29 summary

This notebook prepared dashboard-ready assets for the controlled 50-painting restoration evaluation.

The notebook did not run new experiments. It exported curated CSV and JSON files under:

`outputs/dashboard/`

The dashboard assets include:

- project overview summary,
- dataset and damage summaries,
- model stack summary,
- metric-region policy,
- final model comparison cases,
- model win summaries,
- Stable Diffusion uncertainty cases,
- uncertainty-performance linkage,
- visual case explorer manifest,
- key findings,
- reports manifest,
- dashboard assets manifest.

Main output folders:

- `outputs/dashboard/data/`
- `outputs/dashboard/manifests/`

Main dashboard files:

- `outputs/dashboard/manifests/dashboard_overview_summary.json`
- `outputs/dashboard/manifests/dashboard_assets_manifest.json`
- `outputs/dashboard/manifests/dashboard_key_findings.json`
- `outputs/dashboard/manifests/dashboard_reports_manifest.json`
- `outputs/dashboard/data/dashboard_model_comparison_cases.csv`
- `outputs/dashboard/data/dashboard_uncertainty_cases.csv`
- `outputs/dashboard/data/dashboard_visual_cases.csv`

These files are intended to be loaded by a Streamlit dashboard without requiring the app to parse the full experimental output structure.

The recommended dashboard sections are:

1. Overview,
2. Dataset and damage design,
3. Model stack,
4. Metric-region policy,
5. Final model comparison,
6. Stable Diffusion uncertainty,
7. Visual case explorer,
8. Key findings,
9. Reports and reproducibility.